# Toy generators

This notebooks contains a number of toy-classification generators

In [ ]:
import logging
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as sts

from numpy.typing import NDArray

import ppu
from ppu.generator import Circular, Moons, RingBlobs
from ppu.methods.mlp import MLP
from ppu.methods.tracin import get_random_resampled_tracin, get_random_tracin, get_loss_over_grid, get_proba_over_grid
from ppu.viz import plot_dense_binary_scatter, plot_dense_scatter, plot_ellipse_from_cov, plot_pdf_contours, GridPlot

## Config

In [ ]:
n_samples = 10_000
colors = ppu.viz.set_plot_style(True)
rng = np.random.Generator(np.random.PCG64DXSM(42))

## Definitions

In [ ]:
gen = Moons(class_sep=0.5)
# gen = Circular()
# gen = RingBlobs()

X_train, y_train = gen.rvs(10000)

ax = plot_dense_binary_scatter(X_train, y_train)

In [ ]:
classifier = MLP(hidden_channels=[30, 100, 200, 100, 50, 1])
classifier.fit(X_train, y_train)

grid = GridPlot(X=X_train, y=y_train, n_ticks=1000)

loss_0 = get_loss_over_grid(X=grid.X_grid, y=0, mlp=classifier)
loss_1 = get_loss_over_grid(X=grid.X_grid, y=1, mlp=classifier)

min_loss = np.minimum(loss_0, loss_1)
max_loss = np.maximum(loss_0, loss_1)

In [ ]:
grid.plot(max_loss)

In [ ]:
fig, axs = plt.subplots(figsize=(14, 8), ncols=2, nrows=2)
axs[0, 0].set_title("Maximum loss")
grid.plot(max_loss, overlay=True, ax=axs[0, 0])
axs[0, 1].set_title("Minimum loss")
grid.plot(min_loss, overlay=True, ax=axs[0, 1])
axs[1, 0].set_title("Loss class 0")
grid.plot(loss_0, overlay=True, ax=axs[1, 0])
axs[1, 1].set_title("Loss class 1")
grid.plot(loss_1, overlay=True, ax=axs[1, 1])

## Add some points

In [ ]:
n_extra = 25
label = 1
centroid = (-1.5, 2.5)
scale = 0.2

X_sup = np.empty((X_train.shape[0] + n_extra, X_train.shape[1]), dtype=np.float32)
X_sup[:-n_extra] = X_train
X_sup[-n_extra:] = rng.normal(centroid, scale=scale, size=(n_extra, 2))
y_sup = np.empty(y_train.size + n_extra, dtype=np.float32)
y_sup[:-n_extra] = y_train
y_sup[-n_extra:] = label

plot_dense_binary_scatter(X_sup, y_sup)

In [ ]:
sup_classifier = MLP(hidden_channels=[30, 100, 200, 100, 50, 1], patience=30, frequency=3)
sup_classifier.fit(X_sup, y_sup)

grid = GridPlot(X=X_sup, y=y_sup, n_ticks=1000)

loss_0 = get_loss_over_grid(X=grid.X_grid, y=0, mlp=sup_classifier)
loss_1 = get_loss_over_grid(X=grid.X_grid, y=1, mlp=sup_classifier)

min_loss = np.minimum(loss_0, loss_1)
max_loss = np.maximum(loss_0, loss_1)

fig, axs = plt.subplots(figsize=(14, 8), ncols=2, nrows=2)
axs[0, 0].set_title("Maximum loss")
grid.plot(max_loss, overlay=True, ax=axs[0, 0])
axs[0, 1].set_title("Minimum loss")
grid.plot(min_loss, overlay=True, ax=axs[0, 1])
axs[1, 0].set_title("Loss class 0")
grid.plot(loss_0, overlay=True, ax=axs[1, 0])
axs[1, 1].set_title("Loss class 1")
grid.plot(loss_1, overlay=True, ax=axs[1, 1])

In [ ]:
proba = get_proba_over_grid(X=grid.X_grid, mlp=sup_classifier)

fig, axs = plt.subplots(figsize=(12, 6), ncols=2)
grid.plot(max_loss, ax=axs[0])
grid.plot(proba, ax=axs[1])

## Add the hole

In [ ]:
def create_hole_mask(arr, lb, ub):
    x0 = (arr[:, 0] > lb) & (arr[:, 0]  < ub)
    x1 = (arr[:, 1] > lb) & (arr[:, 1]  < ub)
    return x0 & x1

gen = Circular(class_sep=1.3)

X_train, y_train = gen.rvs(10000)

ax = plot_dense_binary_scatter(X_train, y_train)

n_extra = 100
label = 1
centroid = (0, 0)
scale = 0.02

x_train_mask = create_hole_mask(X_train, -0.1, 0.1)

X_hole = X_train[~x_train_mask, :].copy()
y_hole = y_train[~x_train_mask].copy()

X_sup = np.empty((X_hole.shape[0] + n_extra, X_hole.shape[1]), dtype=np.float32)
X_sup[:-n_extra] = X_hole
X_sup[-n_extra:] = rng.normal(centroid, scale=scale, size=(n_extra, 2))
y_sup = np.empty(y_hole.size + n_extra, dtype=np.float32)
y_sup[:-n_extra] = y_hole
y_sup[-n_extra:] = label

ax = plot_dense_binary_scatter(X_sup, y_sup)

In [ ]:
sup_classifier = MLP(hidden_channels=[30, 100, 200, 100, 50, 1], patience=60, frequency=3)
sup_classifier.fit(X_sup, y_sup)

In [ ]:
grid = GridPlot(X=X_sup, y=y_sup, n_ticks=1000, eps=0.2)

loss_0 = get_loss_over_grid(X=grid.X_grid, y=0, mlp=sup_classifier)
loss_1 = get_loss_over_grid(X=grid.X_grid, y=1, mlp=sup_classifier)

min_loss = np.minimum(loss_0, loss_1)
max_loss = np.maximum(loss_0, loss_1)

fig, axs = plt.subplots(figsize=(14, 8), ncols=2, nrows=2)
axs[0, 0].set_title("Maximum loss")
grid.plot(max_loss, overlay=True, ax=axs[0, 0])
axs[0, 1].set_title("Minimum loss")
grid.plot(min_loss, overlay=True, ax=axs[0, 1])
axs[1, 0].set_title("Loss class 0")
grid.plot(loss_0, overlay=True, ax=axs[1, 0])
axs[1, 1].set_title("Loss class 1")
grid.plot(loss_1, overlay=True, ax=axs[1, 1])

In [ ]:
proba = get_proba_over_grid(X=grid.X_grid, mlp=sup_classifier)

fig, axs = plt.subplots(figsize=(12, 6), ncols=2)
grid.plot(max_loss, ax=axs[0])
grid.plot(proba, ax=axs[1])

## Without hole

In [ ]:
gen = Circular(class_sep=1.3)

X_train, y_train = gen.rvs(10000)

ax = plot_dense_binary_scatter(X_train, y_train)

In [ ]:
n_extra = 300
label = 1
centroid = (0, 0)
scale = 0.02

X_sup = np.empty((X_train.shape[0] + n_extra, X_train.shape[1]), dtype=np.float32)
X_sup[:-n_extra] = X_train
X_sup[-n_extra:] = rng.normal(centroid, scale=scale, size=(n_extra, 2))
y_sup = np.empty(y_train.size + n_extra, dtype=np.float32)
y_sup[:-n_extra] = y_train
y_sup[-n_extra:] = label

ax = plot_dense_binary_scatter(X_sup, y_sup)

In [ ]:
sup_classifier = MLP(hidden_channels=[30, 100, 200, 100, 50, 1], patience=60, frequency=3)
sup_classifier.fit(X_sup, y_sup)

In [ ]:
grid = GridPlot(X=X_sup, y=y_sup, n_ticks=1000, eps=0.2)

loss_0 = get_loss_over_grid(X=grid.X_grid, y=0, mlp=sup_classifier)
loss_1 = get_loss_over_grid(X=grid.X_grid, y=1, mlp=sup_classifier)

min_loss = np.minimum(loss_0, loss_1)
max_loss = np.maximum(loss_0, loss_1)

fig, axs = plt.subplots(figsize=(14, 8), ncols=2, nrows=2)
axs[0, 0].set_title("Maximum loss")
grid.plot(max_loss, overlay=True, ax=axs[0, 0])
axs[0, 1].set_title("Minimum loss")
grid.plot(min_loss, overlay=True, ax=axs[0, 1])
axs[1, 0].set_title("Loss class 0")
grid.plot(loss_0, overlay=True, ax=axs[1, 0])
axs[1, 1].set_title("Loss class 1")
grid.plot(loss_1, overlay=True, ax=axs[1, 1])

In [ ]:
proba = get_proba_over_grid(X=grid.X_grid, mlp=sup_classifier)

fig, axs = plt.subplots(figsize=(12, 6), ncols=2)
grid.plot(max_loss, ax=axs[0])
grid.plot(proba, ax=axs[1])

## Tracin

In [ ]:
gen = Moons(class_sep=0.5)
# gen = Circular()
# gen = RingBlobs()

X_train, y_train = gen.rvs(10000)

ax = plot_dense_binary_scatter(X_train, y_train)

In [ ]:
n_extra = 25
label = 1
centroid = (-1.5, 2.5)
scale = 0.2

X_sup = np.empty((X_train.shape[0] + n_extra, X_train.shape[1]), dtype=np.float32)
X_sup[:-n_extra] = X_train
X_sup[-n_extra:] = rng.normal(centroid, scale=scale, size=(n_extra, 2))
y_sup = np.empty(y_train.size + n_extra, dtype=np.float32)
y_sup[:-n_extra] = y_train
y_sup[-n_extra:] = label

plot_dense_binary_scatter(X_sup, y_sup)

In [ ]:
sup_classifier = MLP(hidden_channels=[30, 100, 200, 100, 50, 1], patience=30, frequency=3)
sup_classifier.fit(X_sup, y_sup)

grid = GridPlot(X=X_sup, y=y_sup, n_ticks=1000)

loss_0 = get_loss_over_grid(X=grid.X_grid, y=0, mlp=sup_classifier)
loss_1 = get_loss_over_grid(X=grid.X_grid, y=1, mlp=sup_classifier)

min_loss = np.minimum(loss_0, loss_1)
max_loss = np.maximum(loss_0, loss_1)

fig, axs = plt.subplots(figsize=(14, 8), ncols=2, nrows=2)
axs[0, 0].set_title("Maximum loss")
grid.plot(max_loss, overlay=True, ax=axs[0, 0])
axs[0, 1].set_title("Minimum loss")
grid.plot(min_loss, overlay=True, ax=axs[0, 1])
axs[1, 0].set_title("Loss class 0")
grid.plot(loss_0, overlay=True, ax=axs[1, 0])
axs[1, 1].set_title("Loss class 1")
grid.plot(loss_1, overlay=True, ax=axs[1, 1])

In [ ]:
classifier = MLP(hidden_channels=[30, 100, 200, 100, 50, 1], patience=30, frequency=3)
classifier.fit(X_train, y_train)

In [ ]:
grid = GridPlot(X=X_sup, y=y_sup, n_ticks=100, eps=0.5)

In [ ]:
scores_1 = get_random_resampled_tracin(
    X_test=grid.X_grid,
    y_test=1,
    X_train=X_train,
    y_train=y_train,
    mlp=classifier,
    batch_size=200,
    n_iter=30
)

scores_0 = get_random_resampled_tracin(
    X_test=grid.X_grid,
    y_test=0,
    X_train=X_train,
    y_train=y_train,
    mlp=classifier,
    batch_size=200,
    n_iter=30
)

In [ ]:
min_scores = np.minimum(scores_0, scores_1)
max_scores = np.maximum(scores_0, scores_1)

fig, axs = plt.subplots(figsize=(14, 8), ncols=2, nrows=2)
axs[0, 0].set_title("Maximum TraceIn")
grid.plot(max_scores[:, -1], overlay=True, ax=axs[0, 0])
axs[0, 1].set_title("Minimum TraceIn")
grid.plot(min_scores[:, -1], overlay=True, ax=axs[0, 1])
axs[1, 0].set_title("TraceIn class 0")
grid.plot(scores_0[:, -1], overlay=True, ax=axs[1, 0])
axs[1, 1].set_title("TraceIn class 1")
grid.plot(scores_1[:, -1], overlay=True, ax=axs[1, 1])
fig.savefig("tracin_moons_30it_square.png", dpi=300)

In [ ]:
n_iters = 30

In [ ]:
fig, axs = plt.subplots(figsize=(24, 26), ncols=5, nrows=6)
axs_ = axs.ravel()
for i in range(n_iters):
    grid.plot(max_scores[:, i], overlay=False, ax=axs_[i])
fig.savefig("max_tracin_iters.png", dpi=300)

In [ ]:
fig, axs = plt.subplots(figsize=(24, 26), ncols=5, nrows=6)
axs_ = axs.ravel()
for i in range(n_iters):
    grid.plot(max_scores[:, i], overlay=True, ax=axs_[i])
fig.savefig("max_tracin_iters_overlay.png", dpi=300)

### Single point

In [ ]:
import torch

In [ ]:
mlp = MLP(hidden_channels=[30, 100, 200, 200, 100, 50, 1], patience=40, frequency=3)
mlp.fit(X_train, y_train)

In [ ]:
x_test = torch.tensor(X_sup[-2], dtype=torch.float32, device=mlp.device)
y_test = torch.tensor((1,), dtype=torch.float32, device=mlp.device)

In [ ]:
batch_size = 400

In [ ]:
x_idx = np.arange(X_train.shape[0])
ridx = rng.choice(x_idx, size=batch_size, replace=False)

In [ ]:
plot_dense_binary_scatter(X_train[ridx], y_train[ridx])

In [ ]:
x_batch = torch.empty((batch_size + 1, 2), device=mlp.device, dtype=torch.float32)
x_batch[:batch_size] = torch.from_numpy(X_train[ridx])
x_batch[-1] = x_test
y_batch = torch.empty(batch_size + 1, device=mlp.device, dtype=torch.float32)
y_batch[:batch_size] = torch.from_numpy(y_train[ridx])
y_batch[-1] = y_test

In [ ]:
plot_dense_binary_scatter(x_batch.numpy(), y_batch.numpy())

In [ ]:
grid = GridPlot(X=X_sup, y=y_sup, n_ticks=1000, eps=0.5)

loss_0 = get_loss_over_grid(X=grid.X_grid, y=0, mlp=mlp)
loss_1 = get_loss_over_grid(X=grid.X_grid, y=1, mlp=mlp)

min_loss = np.minimum(loss_0, loss_1)
max_loss = np.maximum(loss_0, loss_1)
proba = get_proba_over_grid(X=grid.X_grid, mlp=mlp)

fig, axs = plt.subplots(figsize=(12, 6), ncols=2)
grid.plot(max_loss, ax=axs[0])
grid.plot(proba, ax=axs[1])

In [ ]:
n_iters = 40

In [ ]:
initial_loss = mlp.criterion(mlp.model(x_test).squeeze(-1), y_test.squeeze(-1)).detach().cpu().numpy()

point_tracin = np.empty(n_iters)
point_losses = np.empty(n_iters)
point_probas = np.empty(n_iters)
batch_losses = np.empty(shape=(n_iters, batch_size))
shape = (grid.X_grid.shape[0], n_iters)
probas = np.empty(shape=shape, dtype=float)
max_losses = np.empty(shape=shape, dtype=float)


for i in range(n_iters):
    point_grad = mlp.model(x_test).squeeze(-1)
    point_probas[i] = torch.sigmoid(point_grad).detach().cpu().numpy()
    point_loss = mlp.criterion(point_grad, y_test.squeeze(-1)).detach().cpu().numpy()
    point_losses[i] = point_loss
    point_tracin[i] = initial_loss - point_loss
    batch_losses[i, :] = mlp.criterion(mlp.model(x_batch).squeeze(-1), y_batch, reduction="none").detach().cpu().numpy()[:-1]

    loss_0 = get_loss_over_grid(X=grid.X_grid, y=0, mlp=mlp)
    loss_1 = get_loss_over_grid(X=grid.X_grid, y=1, mlp=mlp)

    max_losses[:, i] = np.maximum(loss_0, loss_1).ravel()
    probas[:, i] = get_proba_over_grid(X=grid.X_grid, mlp=mlp).ravel()
    mlp.train_epoch(x_batch, y_batch)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(point_tracin)
ax.set_title("tracin score")
ax.set_ylabel("delta loss")
ax.set_xlabel("iteration")
fig.savefig("tracin_moons_single_point_iter.png", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
_ = ax.plot(batch_losses, c="grey", alpha=0.5, label="batch")
ax.plot(point_losses, c="red", alpha=0.7, label="point")
ax.set_title("loss over iterations")
ax.set_ylabel("loss")
ax.set_xlabel("iteration")
fig.savefig("loss_convergence_moons_single_point_iter.png", dpi=300)

In [ ]:
fig, axs = plt.subplots(figsize=(24, 26), ncols=5, nrows=8)
axs_ = axs.ravel()
for i in range(n_iters):
    grid.plot(max_losses[:, i], overlay=False, ax=axs_[i])
# fig.savefig("max_loss_moons_single_point_iter.png", dpi=300)

In [ ]:
batch_grid = GridPlot(X=X_sup, y=y_sup, n_ticks=1000, eps=0.5)

In [ ]:
batch_grid.X = x_batch.detach().cpu().numpy()
batch_grid.y = y_batch.detach().cpu().numpy()

In [ ]:
fig, axs = plt.subplots(figsize=(24, 26), ncols=5, nrows=8)
axs_ = axs.ravel()
for i in range(n_iters):
    batch_grid.plot(max_losses[:, i], overlay=True, ax=axs_[i])
# fig.savefig("max_loss_moons_single_point_iter_batch_overlay.png", dpi=300)

In [ ]:
fig, axs = plt.subplots(figsize=(24, 26), ncols=5, nrows=8)
axs_ = axs.ravel()
for i in range(n_iters):
    grid.plot(max_losses[:, i], overlay=True, ax=axs_[i])
# fig.savefig("max_loss_moons_single_point_overlay.png", dpi=300)

In [ ]:
fig, axs = plt.subplots(figsize=(24, 26), ncols=5, nrows=8)
axs_ = axs.ravel()
for i in range(n_iters):
    grid.plot(probas[:, i], overlay=False, ax=axs_[i])
# fig.savefig("proba_moons_single_point.png", dpi=300)

Compute gradient

```
import torch

classifier = MLP(hidden_channels=[30, 100, 200, 200, 100, 50, 1], patience=10, frequency=3)
classifier.fit(X_train, y_train)
x_test = torch.tensor((0, 0), dtype=torch.float32, device=classifier.device)
y_test = torch.tensor((0,), dtype=torch.float32, device=classifier.device)
classifier.criterion(classifier.model(x_test), y_test).backward()
list(classifier.model.named_parameters())[-2][1].grad.sum()
```